In [ ]:

!pip install -q transformers datasets evaluate seqeval accelerate pyarrow tensorboard

import os
import glob
import hashlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import evaluate

from datasets import Dataset, DatasetDict, Features, Sequence, Value, load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from transformers import BertTokenizerFast, BertForTokenClassification, BertConfig

from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


SEED = 42
CHUNKSIZE = 300_000
TRAIN_P, VAL_P = 0.8, 0.1

MAX_LEN = 256      
MAX_TOK = 128      

TRAIN_N = 10000
VAL_N = 1000

EPOCHS = 4
LR = 2e-5
BATCH_TRAIN = 16
BATCH_EVAL = 16

MODEL_DIR ="dmis-lab/biobert-base-cased-v1.1"
MODEL_NAME_OUT = "biobert"

OUTDIR = f"./ner_{MODEL_NAME_OUT}_10k"
TB_DIR = f"./tb_{MODEL_NAME_OUT}_10k"
CACHE_DS = "/kaggle/working/hf_ner_dataset_cache"

csv_candidates = glob.glob("/kaggle/input/**/ner_covid_final_data_v3.csv", recursive=True)
if not csv_candidates:
    raise FileNotFoundError("ner_covid_final_data_v3.csv not found in /kaggle/input.")
CSV_PATH = csv_candidates[0]

print("CSV_PATH:", CSV_PATH, "size:", os.path.getsize(CSV_PATH))
print("MODEL_DIR:", MODEL_DIR)
if os.path.isdir(MODEL_DIR):
    print("MODEL files:", sorted(os.listdir(MODEL_DIR))[:30])
else:
    print("MODEL_DIR is Hugging Face model ID:", MODEL_DIR)


def split_bucket(sentence_id: str, seed: int = 42) -> str:
    h = hashlib.md5((str(sentence_id) + str(seed)).encode("utf-8")).hexdigest()
    r = int(h[:8], 16) / 0xFFFFFFFF
    if r < TRAIN_P:
        return "train"
    elif r < TRAIN_P + VAL_P:
        return "validation"
    return "test"

def clean_token(tok: str) -> str:
    if tok is None:
        return ""
    tok = str(tok).strip()
    return "" if tok.lower() == "nan" else tok

def clean_label(lab: str) -> str:
    if lab is None:
        return "Other"
    lab = str(lab).strip()
    if lab == "" or lab.lower() == "nan":
        return "Other"
    if lab == "O":
        return "Other"
    return lab

def bio_fix_sequence(labels):
    fixed = []
    prev = "Other"
    for lab in labels:
        if lab.startswith("I-"):
            ent = lab[2:]
            if prev == "Other" or (prev.startswith(("B-", "I-")) and prev[2:] != ent):
                lab = "B-" + ent
        fixed.append(lab)
        prev = lab
    return fixed

features = Features({
    "tokens": Sequence(Value("string")),
    "ner_tags": Sequence(Value("string")),
})

def sentence_generator(csv_path: str, want_split: str):
    current_sid = None
    tokens_buf, labels_buf = [], []

    reader = pd.read_csv(
        csv_path,
        usecols=["0", "1", "2"],
        dtype={"0": "string", "1": "string", "2": "string"},
        chunksize=CHUNKSIZE,
        keep_default_na=False,
        na_values=[],
    )

    for chunk in reader:
        chunk = chunk.rename(columns={"0": "Token", "1": "Label", "2": "Sentence_ID"})
        chunk["Label"] = chunk["Label"].astype(str).str.strip()

        for tok, lab, sid in zip(chunk["Token"].values, chunk["Label"].values, chunk["Sentence_ID"].values):
            tok = clean_token(tok)
            if tok == "":
                continue

            lab = clean_label(lab)
            sid = str(sid)

            if current_sid is None:
                current_sid = sid

            if sid != current_sid:
                if tokens_buf:
                    labels_fixed = bio_fix_sequence(labels_buf)

                    if len(tokens_buf) > MAX_LEN:
                        tokens_buf = tokens_buf[:MAX_LEN]
                        labels_fixed = labels_fixed[:MAX_LEN]

                    if split_bucket(current_sid, SEED) == want_split:
                        yield {"tokens": tokens_buf, "ner_tags": labels_fixed}

                current_sid = sid
                tokens_buf, labels_buf = [tok], [lab]
            else:
                tokens_buf.append(tok)
                labels_buf.append(lab)

    if tokens_buf and current_sid is not None:
        labels_fixed = bio_fix_sequence(labels_buf)

        if len(tokens_buf) > MAX_LEN:
            tokens_buf = tokens_buf[:MAX_LEN]
            labels_fixed = labels_fixed[:MAX_LEN]

        if split_bucket(current_sid, SEED) == want_split:
            yield {"tokens": tokens_buf, "ner_tags": labels_fixed}


if os.path.exists(CACHE_DS):
    print("Loading cached dataset:", CACHE_DS)
    ds = load_from_disk(CACHE_DS)
else:
    print("Building dataset from CSV...")
    ds = DatasetDict({
        "train": Dataset.from_generator(
            sentence_generator,
            gen_kwargs={"csv_path": CSV_PATH, "want_split": "train"},
            features=features,
        ),
        "validation": Dataset.from_generator(
            sentence_generator,
            gen_kwargs={"csv_path": CSV_PATH, "want_split": "validation"},
            features=features,
        ),
        "test": Dataset.from_generator(
            sentence_generator,
            gen_kwargs={"csv_path": CSV_PATH, "want_split": "test"},
            features=features,
        ),
    })
    ds.save_to_disk(CACHE_DS)
    print("Saved dataset to:", CACHE_DS)

print(ds)

all_labels = sorted({lab for seq in ds["train"]["ner_tags"] for lab in seq})
label2id = {lab: i for i, lab in enumerate(all_labels)}
id2label = {i: lab for lab, i in label2id.items()}

print("num labels:", len(label2id))
print("Has Other:", "Other" in label2id)
print("sample labels:", list(label2id.items())[:10])

tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")

config = BertConfig.from_pretrained("bert-base-cased")
config.num_labels = len(label2id)
config.id2label = id2label
config.label2id = label2id

model = BertForTokenClassification.from_pretrained(
    MODEL_DIR,
    config=config,
    ignore_mismatched_sizes=True
)

print("Loaded model OK")
print("model num_labels:", model.config.num_labels)

ds_small = DatasetDict({
    "train": ds["train"].shuffle(seed=SEED).select(range(min(TRAIN_N, len(ds["train"])))),
    "validation": ds["validation"].shuffle(seed=SEED).select(range(min(VAL_N, len(ds["validation"])))),
})

print(ds_small)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=MAX_TOK,
        is_split_into_words=True,
    )

    aligned = []
    for i, word_labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev = None
        label_ids = []

        for w in word_ids:
            if w is None:
                label_ids.append(-100)
            elif w != prev:
                label_ids.append(label2id[word_labels[w]])
            else:
                label_ids.append(-100)
            prev = w

        aligned.append(label_ids)

    tokenized["labels"] = aligned
    return tokenized

tokenized_small = ds_small.map(
    tokenize_and_align_labels,
    batched=True,
    batch_size=1000,
    remove_columns=["tokens", "ner_tags"],
)

print(tokenized_small)

data_collator = DataCollatorForTokenClassification(tokenizer)
os.environ["TENSORBOARD_LOGGING_DIR"] = TB_DIR

training_args = TrainingArguments(
    output_dir=OUTDIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    num_train_epochs=EPOCHS,
    fp16=torch.cuda.is_available(),

    logging_strategy="steps",
    logging_steps=10,
    logging_first_step=True,
    disable_tqdm=False,
    log_level="info",

    eval_strategy="steps",
    eval_steps=100,
    do_eval=True,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    report_to=["tensorboard"],
    remove_unused_columns=False,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_small["train"],
    eval_dataset=tokenized_small["validation"],
    data_collator=data_collator,
)

trainer.train()

pred_out = trainer.predict(tokenized_small["validation"])

logits = pred_out.predictions
label_ids_eval = pred_out.label_ids
pred_ids = np.argmax(logits, axis=-1)

label_list = [id2label[i] for i in range(model.config.num_labels)]

true_preds = []
true_labs = []

for preds, labs in zip(pred_ids, label_ids_eval):
    p_seq, l_seq = [], []
    for p, l in zip(preds, labs):
        if l == -100:
            continue
        p_seq.append(label_list[int(p)])
        l_seq.append(label_list[int(l)])
    true_preds.append(p_seq)
    true_labs.append(l_seq)

seqeval = evaluate.load("seqeval")
res = seqeval.compute(predictions=true_preds, references=true_labs, zero_division=0)

overall = {
    "precision": float(res["overall_precision"]),
    "recall": float(res["overall_recall"]),
    "f1": float(res["overall_f1"]),
    "accuracy": float(res["overall_accuracy"]),
}
print("Overall:", overall)

rows = []
for ent, v in res.items():
    if ent.startswith("overall_"):
        continue
    if isinstance(v, dict) and "f1" in v:
        rows.append({
            "entity": ent,
            "precision": float(v["precision"]),
            "recall": float(v["recall"]),
            "f1": float(v["f1"]),
            "support": int(v.get("number", v.get("support", 0))),
        })

df = pd.DataFrame(rows)
df_by_support = df.sort_values(["support", "f1"], ascending=[False, False]).reset_index(drop=True)
df_by_f1 = df.sort_values(["f1", "support"], ascending=[False, False]).reset_index(drop=True)

display(df_by_support.head(20))
display(df_by_f1.head(20))

df_by_support.to_csv(f"{MODEL_NAME_OUT}_entity_report_by_support.csv", index=False)
df_by_f1.to_csv(f"{MODEL_NAME_OUT}_entity_report_by_f1.csv", index=False)


log_df = pd.DataFrame(trainer.state.log_history)

train_df = log_df[log_df.get("loss").notna()][["step", "loss"]].copy()
val_df = log_df[log_df.get("eval_loss").notna()][["step", "eval_loss"]].copy()

eval_df_100 = val_df[val_df["step"] % 100 == 0]
train_df_100 = train_df[train_df["step"] % 100 == 0]

table = pd.merge(train_df_100, eval_df_100, on="step", how="outer").sort_values("step")
table = table.rename(columns={"loss": "training_loss", "eval_loss": "validation_loss"})
display(table)


log_df = pd.DataFrame(trainer.state.log_history)

train_df = log_df[log_df.get("loss").notna()][["step","loss"]].copy()
val_df   = log_df[log_df.get("eval_loss").notna()][["step","eval_loss"]].copy()

plt.figure(figsize=(9,4))

plt.plot(train_df["step"], train_df["loss"], linewidth=2)
plt.plot(val_df["step"], val_df["eval_loss"], linewidth=2)

plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("Train vs Validation loss")

plt.legend(["Train","Validation"])

plt.ylim(0.8,3)  

plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig("loss_curve.png", dpi=300)
plt.show()
